# Pretraining on two T4 cards with one client process

Train the base model on both T4 cards of a Kaggle session, then publish the
base stage to the Hugging Face Hub.

One process owns both cards: `MeshRuntime` builds one model replica per device,
one device thread per replica, and reduces the gradients in process
(`docs/client-runtime-design.md`). Training runs in a child process through
`tools/nanochat client`, so the cells stay responsive.

The balanced two-T4 shape is a 252M-parameter model and 500 steps. A full run
takes a few hours and about 9.1 GiB of each 16 GiB card.

This notebook is one of three. Run `eval.ipynb` to measure the published stage,
and `posttrain.ipynb` to fine-tune it. The container and the Hub layout are in
`docs/checkpoint-format.md`.


## Setup

The setup clones the repository, reads the Kaggle secrets, builds the shared
library for the T4 (`sm_75`), and checks the card count. See
`docs/checkpoint-format.md` section 15.


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ming6ao/nanochat.cpp"
REPO = Path("/kaggle/working/nanochat.cpp")
BASE = Path("/kaggle/working/nanochat-cache")

if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
sys.path[:0] = [
    str(REPO / "python"),
    str(REPO / "notebooks"),
    str(REPO / ".agents" / "skills" / "model-fit" / "scripts"),
]


In [ ]:
import session

sess = session.setup(REPO_URL, REPO, arch="sm_75", devices=2, base=BASE)
print("library:", sess.library)
print("secrets:", sorted(sess.secrets))


In [ ]:
import nanochat_cpp as nc

for device in nc.devices():
    print(device, device.name,
          f"{device.total_memory_bytes / 2**30:.1f} GiB")


## Data and tokenizer

Stage one ClimbMix train shard and one validation shard, then pull the
tokenizer from the Hub. The tokenizer lives in
`<namespace>/nanochat-tokenizer` as one file for each vocabulary size. A pull
miss trains the tokenizer and pushes it. The download is the one fixed cost.


In [ ]:
import os
import shutil

import hub_checkpoint

VOCAB_SIZE = 32768
DATA = BASE / "base_data_climbmix"
TRAIN_SHARD = "shard_00000.parquet"
VAL_SHARD = "shard_06542.parquet"

if not (DATA / VAL_SHARD).is_file():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "huggingface_hub"], check=True)
    from huggingface_hub import snapshot_download

    snapshot_download(repo_id="karpathy/climbmix-400b-shuffle",
                      repo_type="dataset", local_dir=str(DATA),
                      allow_patterns=[TRAIN_SHARD, VAL_SHARD],
                      token=os.environ.get("HF_TOKEN"))

TOKENIZER = BASE / "tokenizer" / "tokenizer.nctoken"
TOKENIZER.parent.mkdir(parents=True, exist_ok=True)
TOKENIZER_REPO = hub_checkpoint.repo_id_for("tokenizer")

if not TOKENIZER.is_file():
    pulled = hub_checkpoint.pull_tokenizer(
        TOKENIZER_REPO, VOCAB_SIZE, TOKENIZER.parent,
        token=os.environ.get("HF_TOKEN"))
    if pulled is not None:
        shutil.copyfile(pulled, TOKENIZER)
        print("pulled tokenizer:", pulled)
    else:
        print("no tokenizer on the Hub; training one")
        nc.toolchain.build(["//src/tokenizer:tok_train_main"])
        nc.toolchain.run("t0-cpu", [
            "bazel-bin/src/tokenizer/tok_train_main",
            "--parquet", f"{DATA}/{TRAIN_SHARD}",
            "--vocab-size", str(VOCAB_SIZE),
            "--max-chars", "2000000000",
            "--out", str(TOKENIZER),
        ])
        hub_checkpoint.push_tokenizer(TOKENIZER_REPO, VOCAB_SIZE, TOKENIZER,
                                      token=os.environ.get("HF_TOKEN"))

tokenizer = nc.Tokenizer.load(str(TOKENIZER))
assert tokenizer.decode(tokenizer.encode("The capital of France is")) == \
    "The capital of France is", "the tokenizer round trip failed"
print("train:", DATA / TRAIN_SHARD)
print("tokenizer vocab:", tokenizer.vocab_size)


## Model

Change the knobs to try another shape. The default is the balanced two-T4
shape: 16 layers, hidden 1024, 8 heads, 4 key/value heads, and 4096 tokens,
about 252M parameters. Keep `num_heads` and `num_kv_heads` as divisors of
`hidden_dim`.


In [ ]:
import args
import model_fit

MODEL = args.ModelArgs(num_layers=16, hidden_dim=1024, num_heads=8,
                       num_kv_heads=4, seq_len=4096, vocab_size=VOCAB_SIZE)

spec = model_fit.ModelSpec(
    layers=MODEL.num_layers, hidden=MODEL.hidden_dim, heads=MODEL.num_heads,
    kv_heads=MODEL.num_kv_heads, seq=MODEL.seq_len, vocab=MODEL.vocab_size,
    window=MODEL.window_pattern, value_embedding=MODEL.value_embedding)
config = MODEL.to_config()

counts = model_fit.parameter_counts(spec)
t4 = model_fit.hardware("t4")
print("parameters:", f"{counts['total']:,}")
print("flops per token:", f"{model_fit.flops_per_token(spec) / 1e9:.3f} GFLOP")
print("estimated card memory at per-rank batch 1:",
      f"{model_fit.card_bytes(spec, 1, 'fp32') / 2**30:.2f} GiB of 16 GiB")


## Train

The runtime takes the global batch. `per_rank_batch * grad_accum * world_size`
is the global sequence count, split across the two cards. The gradients reduce
once per step.

The cell builds the plan, starts the worker in a child process, and streams the
per-step log. Interrupt the cell to stop the worker; it saves the checkpoint
first.


In [ ]:
import time

import mesh_train

RUNS = Path("/kaggle/working/runs")
BASE_STEPS = 500
RUN_NAME = (f"base-L{MODEL.num_layers}-H{MODEL.hidden_dim}"
            f"-n{MODEL.num_heads}-kv{MODEL.num_kv_heads}"
            f"-s{MODEL.seq_len}")

plan = mesh_train.MeshTrainPlan(
    train_parquet=f"{DATA}/{TRAIN_SHARD}",
    tokenizer=str(TOKENIZER),
    out_dir=str(RUNS / RUN_NAME),
    config=MODEL.to_config(),
    per_rank_batch=1, grad_accum=8, world_size=2,
    steps=BASE_STEPS, seed=42, data_seed=42,
    log_every=1, save_every=50,
    preset="track3", resume=True,
    flops_per_token=model_fit.flops_per_token(spec),
    peak_flops=t4.peak_flops("fp32"))
print("run:", plan.out_dir, "| tokens per step:", plan.tokens_per_step)

live = mesh_train.launch(plan, root=REPO, profile="train")
log_path = live.log_path
position = 0
try:
    while live.running():
        text = log_path.read_text(encoding="utf-8", errors="replace") \
            if log_path.is_file() else ""
        if len(text) > position:
            print(text[position:], end="", flush=True)
            position = len(text)
        time.sleep(5)
except KeyboardInterrupt:
    # Stop the worker, then finish this cell cleanly. `terminate` ignores a
    # second interrupt.
    print("\ninterrupt received; stopping the worker", flush=True)
    try:
        live.terminate()
    except KeyboardInterrupt:
        pass

returncode = live.wait()
print("worker exit code:", returncode)
if returncode != 0:
    print(live.tail(60))

# The per-step log streamed above. This block prints the run summary.
records = mesh_train.read_metrics(plan.metrics_path())
summary = mesh_train.read_summary(plan.out_dir)
print(f"steps {len(records)}  "
      f"mean loss {mesh_train.mean_loss(records):.4f}  "
      f"steady MFU {mesh_train.steady_mfu(records) * 100:.2f}%")
mfu = mesh_train.steady_mfu(records)
assert mfu >= 0.30, f"steady-state MFU {mfu * 100:.2f}% is below 30%"
if summary:
    digests = summary["replica_digests"]
    print("replica digests:", digests)
    assert len(digests) == 2 and len(set(digests)) == 1, (
        "the two replicas differ: the gradient reduce is broken")
checkpoint = Path(summary.get("checkpoint") or plan.checkpoint_path())
if checkpoint.is_file():
    print("checkpoint:", checkpoint, checkpoint.stat().st_size, "bytes")


## Publish

Write the model arguments, the training job, the run record, the tokenizer, and
the model card beside the checkpoint, then upload the `base` stage and tag it.


In [ ]:
import datetime
import json
import shutil

run_dir = Path(plan.out_dir)
run_dir.mkdir(parents=True, exist_ok=True)

(run_dir / "model.json").write_text(MODEL.to_json() + "\n",
                                    encoding="utf-8")
(run_dir / "train.json").write_text(
    json.dumps(plan.job(), indent=1, sort_keys=True) + "\n", encoding="utf-8")
shutil.copyfile(TOKENIZER, run_dir / "tokenizer.nctoken")

git_revision = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True).stdout.strip()
record = hub_checkpoint.build_run_record(
    stage="base", run_name=RUN_NAME, git_revision=git_revision,
    build_key=sess.build_key, cuda_arch="sm_75", step=BASE_STEPS,
    created_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
Path(run_dir, "run.json").write_text(
    json.dumps(record, indent=1, sort_keys=True) + "\n", encoding="utf-8")

card = f"""# {RUN_NAME}

Base model, pretrained on ClimbMix for {BASE_STEPS} steps.

- architecture: `{MODEL.to_json()}`
- container: `{hub_checkpoint.CONTAINER}`
- source revision: `{git_revision}`
"""
Path(run_dir, "README.md").write_text(card, encoding="utf-8")
print("run directory:", run_dir)
print(record)


In [ ]:
REPO_ID = hub_checkpoint.repo_id_for(RUN_NAME)
hub_checkpoint.push_stage(REPO_ID, "base", run_dir,
                          token=os.environ.get("HF_TOKEN"),
                          tag=f"base-{BASE_STEPS}")
print("pushed:", REPO_ID, "stage base", f"tag base-{BASE_STEPS}")


## Sanity check

Load the checkpoint into one `nc.Model` on card 0, measure a few validation
steps, and print two greedy samples.


In [ ]:
model = nc.Model(config, device="cuda", seed=0)
model.load(str(plan.checkpoint_path()))
print("device:", model.device, "parameters:", f"{model.param_total():,}")

report = nc.evaluate(model, parquet=str(DATA / VAL_SHARD), tokenizer=tokenizer,
                     tokens=config.seq_len * 4, batch=1, seed=43)
print(f"validation bits per byte: {report.bpb:.4f}")

bos = tokenizer.get_bos_token_id()
for prompt in ["The capital of France is", "The chemical symbol of gold is"]:
    prompt_ids = ([bos] if bos >= 0 else []) + tokenizer.encode(prompt)
    row = model.generate(prompt_ids, max_tokens=16, temperature=0.0)[0]
    generated = [token for token, keep in zip(row.tokens, row.mask) if keep]
    print("-" * 40)
    print(prompt + tokenizer.decode(generated))


## Next

Run `eval.ipynb` in a later session. It downloads the tag `base-500` and
measures the base stage. Run `posttrain.ipynb` to fine-tune it into the `sft`
stage.
